# Train an NFL win-probability model

This notebook trains a ML model to predict live win probability in an NFL game. It loads the downloaded nflverse play-by-play data, creates a binary target, engineers pre-play game-state features, trains an XGBoost classifier, evaluates it with `evals.py`, and saves a model artifact.

Current split is: 1999–2021 train, 2022–2023 validation, and 2024–2025 test. 

The 2026 file is intentionally held out for now. 

In [ ]:
from pathlib import Path
import json
import sys
import warnings

###################################################
######### Necessary packages to run model #########
###################################################

import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBClassifier

def find_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'pbp').exists() and (candidate / 'evals.py').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from the project root or its notebooks/ directory.')

ROOT = find_root()
sys.path.insert(0, str(ROOT))
from evals import evaluate_predictions

DATA_DIR = ROOT / 'data' / 'raw' / 'pbp'
RESULTS_DIR = ROOT / 'results'
ARTIFACT_DIR = ROOT / 'artifacts'
RANDOM_STATE = 42
USE_SPREAD = True

TRAIN_SEASONS = list(range(1999, 2022))
VALIDATION_SEASONS = [2022, 2023]
TEST_SEASONS = [2024, 2025]

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
print(f'Project root: {ROOT}')
print(f'Split: train={TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]}, validation={VALIDATION_SEASONS}, test={TEST_SEASONS}')

## 1. Preprocessing: load PBP and define the target

The target is whether the team with possession on a play eventually wins the game. `result` is the final home-team score margin, so the target is derived from the possession team and final result. Ties, malformed possession rows, and rows without a usable game clock are removed. 

In [ ]:
RAW_COLUMNS = [
    'play_id', 'game_id', 'season', 'season_type',
    'home_team', 'away_team', 'posteam',
    'game_seconds_remaining', 'half_seconds_remaining',
    'score_differential', 'yardline_100', 'down', 'ydstogo', 'qtr',
    'goal_to_go', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining',
    'spread_line', 'roof', 'result'
]

ALL_SEASONS = TRAIN_SEASONS + VALIDATION_SEASONS + TEST_SEASONS

paths = [DATA_DIR / f'play_by_play_{season}.parquet' for season in ALL_SEASONS]
missing = [path for path in paths if not path.exists()]

if missing:
    raise FileNotFoundError('Missing PBP file(s): ' + ', '.join(str(path) for path in missing))

pieces = []
for season, path in zip(ALL_SEASONS, paths):
    frame = pd.read_parquet(path, columns=RAW_COLUMNS)
    frame['season'] = season
    pieces.append(frame)

pbp = pd.concat(pieces, ignore_index=True)
pbp = pbp[pbp['season_type'].isin(['REG', 'POST'])].copy()
pbp['result'] = pd.to_numeric(pbp['result'], errors='coerce')
pbp = pbp[pbp['result'].notna() & (pbp['result'] != 0)]
pbp = pbp[pbp['game_seconds_remaining'].notna()].copy()

is_home_possession = pbp['posteam'].eq(pbp['home_team'])
is_away_possession = pbp['posteam'].eq(pbp['away_team'])

pbp['target'] = np.select(
    [is_home_possession, is_away_possession],
    [pbp['result'].gt(0), pbp['result'].lt(0)],
    default=np.nan,
).astype('float32')

pbp = pbp[pbp['target'].notna()].copy()
pbp = pbp.sort_values(['season', 'game_id', 'play_id']).reset_index(drop=True)

print(f'Rows after preprocessing: {len(pbp):,}')
print(f'Games: {pbp.game_id.nunique():,}')
print(f'Possession-team win rate: {pbp.target.mean():.4f}')

## 2. Feature engineering: pre-play game state

Features that the model will use to predict win probability

In [ ]:
STATE_DEFAULTS = {
    'game_seconds_remaining': 0.0,
    'half_seconds_remaining': 0.0,
    'score_differential': 0.0,
    'yardline_100': 50.0,
    'down': 0.0,
    'ydstogo': 10.0,
    'qtr': 4.0,
    'posteam_timeouts_remaining': 0.0,
    'defteam_timeouts_remaining': 0.0,
}

def make_features(frame, include_spread=True):
    features = pd.DataFrame(index=frame.index)
    numeric = {}
    for column, default in STATE_DEFAULTS.items():
        values = pd.to_numeric(frame[column], errors='coerce')
        features[f'{column}_missing'] = values.isna().astype('float32')
        numeric[column] = values.fillna(default).astype('float32')
        features[column] = numeric[column]

    features['home_possession'] = frame['posteam'].eq(frame['home_team']).astype('float32')
    features['seconds_remaining_fraction'] = numeric['game_seconds_remaining'] / 3600.0
    features['half_seconds_remaining_fraction'] = numeric['half_seconds_remaining'] / 1800.0
    features['score_differential_abs'] = numeric['score_differential'].abs()
    features['score_time_interaction'] = (numeric['score_differential'] * (1.0 - features['seconds_remaining_fraction']))

    goal_to_go = frame['goal_to_go'].fillna(False).astype(bool)
    features['goal_to_go'] = goal_to_go.astype('float32')

    spread = pd.to_numeric(frame['spread_line'], errors='coerce')
    features['spread_available'] = spread.notna().astype('float32')

    if include_spread:
        features['spread_line'] = spread.fillna(0.0).astype('float32')

    roof = frame['roof'].fillna('unknown').astype(str).str.lower()
    roof = roof.replace({'open': 'outdoors', 'closed': 'retractable'})

    features = pd.concat([features, roof.rename('roof')], axis=1)
    features = pd.get_dummies(features, columns=['roof'], dtype='float32')

    return features.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype('float32')

features = make_features(pbp, include_spread=USE_SPREAD)

print(f'Feature matrix shape: {features.shape}')

## 3. Feature engineering: season-based matrices

A season-based split prevents plays from the same season appearing in both training and evaluation. Keep `X_test` untouched until the final evaluation cell.

In [ ]:
train_mask = pbp['season'].isin(TRAIN_SEASONS)
validation_mask = pbp['season'].isin(VALIDATION_SEASONS)
test_mask = pbp['season'].isin(TEST_SEASONS)

X_train = features.loc[train_mask].copy()
X_validation = features.loc[validation_mask].copy()
X_test = features.loc[test_mask].copy()
y_train = pbp.loc[train_mask, 'target'].astype('float32')
y_validation = pbp.loc[validation_mask, 'target'].astype('float32')
y_test = pbp.loc[test_mask, 'target'].astype('float32')
FEATURE_COLUMNS = list(X_train.columns)

print('Rows:', {'train': len(X_train), 'validation': len(X_validation), 'test': len(X_test)})
print(f'Features: {len(FEATURE_COLUMNS)}')

## 4. XGBoost training

This is the main model-change cell. Change the feature function, hyperparameters, or split above, then rerun from the top. Early stopping uses only the validation seasons; the test seasons remain untouched.

In [ ]:
model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method='hist',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    early_stopping_rounds=25,
)

#Tuning parameters might be helpful 

model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=False,
)

best_iteration = getattr(model, 'best_iteration', model.n_estimators - 1)

print(f'Best boosting iteration: {best_iteration}')
print(f'Validation log loss: {model.best_score:.5f}' if hasattr(model, 'best_score') else 'Validation complete')

## 5. Evaluation through `evals.py`

The evaluation module reports Brier Skill Score (BSS), ROC AUC, and KS. It also plots expected win probability against actual win rate, with the 45-degree line representing perfect calibration. BSS uses the training-season win rate as its reference probability.

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics, calibration, calibration_figure = evaluate_predictions(
    y_test.to_numpy(),
    test_probability,
    reference_prob=float(y_train.mean()),
    title='XGBoost NFL win probability: expected vs. actual',
    save_path=RESULTS_DIR / 'xgb_expected_vs_actual.png',
    show=True,
)

display(pd.DataFrame([metrics])[['BSS', 'AUC', 'KS', 'brier_score', 'log_loss', 'n']])
pd.DataFrame([metrics]).to_csv(RESULTS_DIR / 'xgb_test_metrics.csv', index=False)
calibration.to_csv(RESULTS_DIR / 'xgb_test_calibration.csv', index=False)

## 6. Save a pushable model artifact

The model JSON and feature manifest make a successful experiment reproducible. A future change can be evaluated, inspected, and then committed together with its metrics and calibration plot.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / 'wp_xgboost.json'
feature_manifest_path = ARTIFACT_DIR / 'feature_schema.json'
model.save_model(model_path)
feature_manifest = {
    'feature_columns': FEATURE_COLUMNS,
    'use_spread': USE_SPREAD,
    'train_seasons': TRAIN_SEASONS,
    'validation_seasons': VALIDATION_SEASONS,
    'test_seasons': TEST_SEASONS,
    'random_state': RANDOM_STATE,
    'best_iteration': int(best_iteration),
}
feature_manifest_path.write_text(json.dumps(feature_manifest, indent=2))
print(f'Saved model: {model_path}')
print(f'Saved feature manifest: {feature_manifest_path}')
print(f'Saved metrics and plot under: {RESULTS_DIR}')